In [1]:
import os, json, re
import pandas as pd
import numpy as np
import joblib
from datetime import datetime
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.multioutput import MultiOutputClassifier

# ── Chemins ───────────────────────────────────────────────────────────────
root       = r'C:\Users\HP\OneDrive\Google Drive\BIG DATA\DATA SCIENCE\Projets\Energy-Data-Scientist\predictive-maintenance-electrical-grid'
data_path  = os.path.join(root, 'data', 'predictive_maintenance.csv')
save_dir   = os.path.join(root, 'maintenance_predictive', 'models')

# ── Chargement des données brutes ─────────────────────────────────────────
df = pd.read_csv(data_path)
print(f"Dataset chargé : {df.shape}")
print(f"Colonnes : {df.columns.tolist()}")

# ── Feature engineering ───────────────────────────────────────────────────
df['temp_diff']      = df['Process temperature [K]'] - df['Air temperature [K]']
df['power_kw']       = (df['Torque [Nm]'] * df['Rotational speed [rpm]'] * 2 * np.pi / 60) / 1000
df['thermal_stress'] = df['temp_diff'] * df['power_kw']
df['Type_encoded']   = df['Type'].map({'L': 0, 'M': 1, 'H': 2})

# ── Features communes ─────────────────────────────────────────────────────
feature_cols  = ['Air temperature [K]', 'Process temperature [K]',
                 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]',
                 'temp_diff', 'power_kw', 'thermal_stress', 'Type_encoded']
labels        = ['TWF', 'HDF', 'PWF', 'OSF', 'RNF']
cols_to_drop  = ['Machine failure', 'UDI', 'Product ID'] + labels

X = df.drop(columns=[c for c in cols_to_drop if c in df.columns])
X = X[feature_cols]

# ══════════════════════════════════════════════════════════════════════════
# NIVEAU 1 — Prédiction de panne (binaire)
# ══════════════════════════════════════════════════════════════════════════
y = df['Machine failure']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scaling — fit sur train uniquement
scaler_n1    = RobustScaler()
X_train_s    = pd.DataFrame(scaler_n1.fit_transform(X_train), columns=feature_cols)
X_test_s     = pd.DataFrame(scaler_n1.transform(X_test),      columns=feature_cols)

# Entraînement Gradient Boosting (modèle retenu)
gb_n1 = GradientBoostingClassifier(
    n_estimators=150, learning_rate=0.05, max_depth=4, random_state=42
)
gb_n1.fit(X_train_s, y_train)
print(f"✓ Niveau 1 entraîné — score test : {gb_n1.score(X_test_s, y_test):.4f}")

# Sauvegarde
joblib.dump(gb_n1,     os.path.join(save_dir, 'gradient_boosting_niveau1.pkl'))
joblib.dump(scaler_n1, os.path.join(save_dir, 'robust_scaler_niveau1.pkl'))

# Config JSON
config_n1 = {
    "nom_modele"    : "GradientBoostingClassifier",
    "date_entrainement": datetime.now().strftime("%Y-%m-%d"),
    "python_version": "3.11",
    "sklearn_version": "1.4.2",
    "hyperparametres": {"n_estimators": 150, "learning_rate": 0.05, "max_depth": 4},
    "seuil_decision": 0.4236,
    "features_utilisees": feature_cols,
    "metriques_test": {
        "precision_panne": 0.93,
        "recall_panne"   : 0.82,
        "f1_panne"       : 0.88,
        "auc_roc"        : 0.9798
    }
}
with open(os.path.join(save_dir, 'config_niveau1.json'), 'w', encoding='utf-8') as f:
    json.dump(config_n1, f, indent=4, ensure_ascii=False)

print("✓ Niveau 1 sauvegardé (gradient_boosting_niveau1.pkl, robust_scaler_niveau1.pkl, config_niveau1.json)")

# ══════════════════════════════════════════════════════════════════════════
# NIVEAU 2 — Identification du mode de panne (multi-label)
# ══════════════════════════════════════════════════════════════════════════
Y = df[labels]

X_train2, X_test2, Y_train2, Y_test2 = train_test_split(
    X, Y, test_size=0.2, random_state=42, stratify=df['Machine failure']
)

# Scaling Niveau 2
numeric_cols_ml = ['Air temperature [K]', 'Process temperature [K]',
                   'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]',
                   'temp_diff', 'power_kw', 'thermal_stress']

scaler_n2 = RobustScaler()
X_train2[numeric_cols_ml] = scaler_n2.fit_transform(X_train2[numeric_cols_ml])
X_test2[numeric_cols_ml]  = scaler_n2.transform(X_test2[numeric_cols_ml])

# Renommage des colonnes (requis par les estimateurs)
def clean_col_names(df_):
    df_ = df_.copy()
    df_.columns = [re.sub(r'[\[\]<>]', '', c).strip().replace(' ', '_')
                   for c in df_.columns]
    return df_

X_train2_clean = clean_col_names(X_train2)
X_test2_clean  = clean_col_names(X_test2)
feature_cols_clean = list(X_train2_clean.columns)

# Entraînement MultiOutput Gradient Boosting
gb_ml = MultiOutputClassifier(
    GradientBoostingClassifier(
        n_estimators=150, learning_rate=0.05, max_depth=4, random_state=42
    ),
    n_jobs=-1
)
gb_ml.fit(X_train2_clean, Y_train2)
print(f"✓ Niveau 2 entraîné — {len(gb_ml.estimators_)} estimateurs")

# Sauvegarde
joblib.dump(gb_ml,    os.path.join(save_dir, 'gb_multilabel_niveau2.pkl'))
joblib.dump(scaler_n2, os.path.join(save_dir, 'robust_scaler_niveau2.pkl'))

# Config JSON
config_n2 = {
    "nom_modele"    : "MultiOutputClassifier(GradientBoostingClassifier)",
    "date_entrainement": datetime.now().strftime("%Y-%m-%d"),
    "python_version": "3.11",
    "sklearn_version": "1.4.2",
    "labels"        : labels,
    "features_utilisees": feature_cols_clean,
    "preprocessing" : {
        "scaler"          : "RobustScaler",
        "features_scalees": numeric_cols_ml
    },
    "seuils_production": {
        "TWF": None,
        "HDF": 0.7846,
        "PWF": 0.9998,
        "OSF": 0.2058,
        "RNF": None
    }
}
with open(os.path.join(save_dir, 'config_niveau2.json'), 'w', encoding='utf-8') as f:
    json.dump(config_n2, f, indent=4, ensure_ascii=False)

print("✓ Niveau 2 sauvegardé (gb_multilabel_niveau2.pkl, robust_scaler_niveau2.pkl, config_niveau2.json)")

# ── Vérification finale ───────────────────────────────────────────────────
print("\n═══ VÉRIFICATION FINALE ═══")
for f in os.listdir(save_dir):
    taille = os.path.getsize(os.path.join(save_dir, f)) / 1024
    print(f"  {f} ({taille:.1f} Ko)")

import sklearn
print(f"\nPython : {__import__('sys').version}")
print(f"scikit-learn : {sklearn.__version__}")
print("\n✅ Modèles compatibles Python 3.11 + scikit-learn 1.4.2 — prêts pour Streamlit Cloud")


Dataset chargé : (10000, 14)
Colonnes : ['UDI', 'Product ID', 'Type', 'Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]', 'Machine failure', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF']
✓ Niveau 1 entraîné — score test : 0.9915
✓ Niveau 1 sauvegardé (gradient_boosting_niveau1.pkl, robust_scaler_niveau1.pkl, config_niveau1.json)
✓ Niveau 2 entraîné — 5 estimateurs
✓ Niveau 2 sauvegardé (gb_multilabel_niveau2.pkl, robust_scaler_niveau2.pkl, config_niveau2.json)

═══ VÉRIFICATION FINALE ═══
  config_niveau1.json (0.7 Ko)
  config_niveau2.json (1.0 Ko)
  gb_multilabel_niveau2.pkl (1274.8 Ko)
  gradient_boosting_niveau1.pkl (315.5 Ko)
  robust_scaler_niveau1.pkl (1.0 Ko)
  robust_scaler_niveau2.pkl (1.0 Ko)

Python : 3.11.15 | packaged by Anaconda, Inc. | (main, Jun 11 2026, 15:12:53) [MSC v.1942 64 bit (AMD64)]
scikit-learn : 1.4.2

✅ Modèles compatibles Python 3.11 + scikit-learn 1.4.2 — prêts pour Streamlit Cloud
